# Reflection Loop

> Notebook generated from [`examples/patterns/06_reflection_loop.py`](../../examples/patterns/06_reflection_loop.py).

| Field | Value |
|------|-------|
| **Dataset** | Writing Prompts (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Iterations of generate→critique→refine with their score; demo of the `@with_reflection` decorator.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Reflection Loop — Iterative improvement of technical articles
=========================================================
Pattern: SPEC-035 / prismal.agents.patterns.reflection

Dataset: Writing Prompts + technical-documentation prompts
  • FairytaleQA / Writing Prompts from Reddit (r/WritingPrompts).
  • Reference: https://huggingface.co/datasets/euclaise/writingprompts
  • Why: The Reflection pattern requires a domain where quality
    is gradually improvable: technical writing and articles
    are ideal because there are objective criteria (clarity, accuracy,
    completeness) that guide the refinement iterations.

Pattern description:
  1. generate_fn produces a draft.
  2. critique_fn evaluates the draft and returns (feedback, score ∈ [0,1]).
  3. If score >= threshold → return the draft.
  4. Otherwise → generate_fn receives the previous draft and the critique to refine.
  5. Up to max_iterations iterations; returns the best draft observed.

Also demonstrates the @with_reflection decorator for LangGraph nodes.

Usage:
    uv run python examples/patterns/06_reflection_loop.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.messages import HumanMessage, SystemMessage

from prismal.agents.patterns.reflection import reflection_loop
from prismal.agents.state import AgentState, initial_state
from prismal.core.config import get_settings
from prismal.providers.registry import ProviderRegistry

## Dataset: technical writing prompts

In [ ]:
WRITING_PROMPTS = [
    {
        "id": "WP1",
        "topic": "Transformer explanation for developers",
        "instruction": (
            "Write a 3-4 paragraph explanation of the Transformer architecture "
            "for a developer with Python experience but no deep learning "
            "background. Use concrete analogies."
        ),
        "evaluation_criteria": [
            "concrete and understandable analogies",
            "no unexplained mathematical jargon",
            "mention of attention and encoder/decoder",
            "practical application examples",
        ],
    },
    {
        "id": "WP2",
        "topic": "async/await guide in Python",
        "instruction": (
            "Write a concise guide to async/await in Python for a programmer "
            "coming from JavaScript. Explain the key differences and the most "
            "common mistakes. Include at least one code example."
        ),
        "evaluation_criteria": [
            "explicit comparison with JavaScript",
            "working code example",
            "common mistakes mentioned",
            "asyncio.run() or loop.run_until_complete() included",
        ],
    },
    {
        "id": "WP3",
        "topic": "Introduction to RAG for product managers",
        "instruction": (
            "Write an explanation of Retrieval-Augmented Generation (RAG) "
            "for a product manager with no technical background. Explain what it is, "
            "what it is for, and when to use it vs fine-tuning. Maximum 300 words."
        ),
        "evaluation_criteria": [
            "no excessive technical jargon",
            "comparison RAG vs fine-tuning",
            "concrete use cases",
            "length <= 300 words",
        ],
    },
]

## Reflection loop callables

In [ ]:
async def generate_article(
    state: AgentState,
    previous_draft: str | None = None,
    critique: str | None = None,
) -> str:
    """Generate or improve a technical article.

    If previous_draft and critique are provided, the LLM refines the draft.
    Otherwise, it generates from scratch.

    Args:
        state: AgentState with 'instruction' in metadata.
        previous_draft: Previous draft (if there was a prior iteration).
        critique: Feedback from the previous evaluation.

    Returns:
        New draft as a string.
    """
    settings = get_settings()
    llm = ProviderRegistry(settings=settings).get_llm()

    instruction = state.get("metadata", {}).get("instruction", "Write about AI")

    if previous_draft and critique:
        system = (
            "You are an expert technical writer. Your previous draft was evaluated "
            "and received the following feedback. Rewrite the article incorporating "
            "the suggested improvements. Keep what already works well."
        )
        user = (
            f"Original instruction: {instruction}\n\n"
            f"Your previous draft:\n{previous_draft}\n\n"
            f"Feedback received:\n{critique}\n\n"
            "Please rewrite the article improving the points indicated."
        )
    else:
        system = (
            "You are an expert technical writer in technology and programming. "
            "Write clear, concise and useful articles for the target audience."
        )
        user = f"Instruction: {instruction}"

    response = await llm.ainvoke([SystemMessage(content=system), HumanMessage(content=user)])
    return str(response.content).strip()


async def critique_article(
    draft: str,
    state: AgentState,
) -> tuple[str, float]:
    """Evaluate the quality of a technical draft.

    Checks the criteria defined in state["metadata"]["criteria"].

    Args:
        draft: Text of the draft to evaluate.
        state: AgentState with 'criteria' in metadata.

    Returns:
        Tuple (feedback: str, score: float in [0,1]).
    """
    settings = get_settings()
    llm = ProviderRegistry(settings=settings).get_llm()

    criteria = state.get("metadata", {}).get("criteria", [])
    criteria_text = "\n".join(f"  - {c}" for c in criteria)

    system = (
        "You are an expert technical editor. Evaluate the text against the given "
        "criteria. Provide:\n"
        "1. A score from 0.0 to 1.0 (on the FIRST line, only the number).\n"
        "2. Specific and actionable feedback on what to improve.\n"
        "Be strict: a score of 0.9+ only if the text is excellent."
    )
    user = (
        f"Evaluation criteria:\n{criteria_text}\n\n"
        f"Text to evaluate:\n{draft}\n\n"
        "Response format:\n"
        "[score between 0.0 and 1.0]\n"
        "[detailed feedback]"
    )

    response = await llm.ainvoke([SystemMessage(content=system), HumanMessage(content=user)])

    raw = str(response.content).strip()
    lines = raw.split("\n", 1)

    # Parse score
    try:
        score = float(lines[0].strip())
        score = max(0.0, min(1.0, score))
    except ValueError:
        # Fallback: find the first floating-point number
        import re

        numbers = re.findall(r"\b0\.\d+\b|\b1\.0\b", raw)
        score = float(numbers[0]) if numbers else 0.5

    feedback = lines[1].strip() if len(lines) > 1 else raw

    return feedback, score


## Main function

In [ ]:
async def run_reflection(prompt: dict, threshold: float = 0.85) -> None:
    """Run the reflection loop for a writing prompt.

    Args:
        prompt: Dictionary with topic, instruction and evaluation_criteria.
        threshold: Minimum score to accept the draft.
    """
    print(f"\n[{prompt['id']}] Topic: {prompt['topic']}")
    print(f"  Quality threshold: {threshold}")
    print(f"  Criteria: {len(prompt['evaluation_criteria'])} criteria")

    # Prepare the state
    state: AgentState = initial_state()
    state["metadata"]["instruction"] = prompt["instruction"]
    state["metadata"]["criteria"] = prompt["evaluation_criteria"]
    state["metadata"]["topic"] = prompt["topic"]

    # Run the reflection loop
    best_draft, final_score = await reflection_loop(
        generate_fn=generate_article,
        critique_fn=critique_article,
        state=state,
        threshold=threshold,
        max_iterations=3,
    )

    print(f"\n  Final score: {final_score:.3f}")
    print(f"  Final draft ({len(best_draft)} chars):")
    print(f"  {'─' * 60}")
    # Show first 400 characters of the draft
    preview = best_draft[:400]
    for line in preview.split("\n"):
        print(f"  {line}")
    if len(best_draft) > 400:
        print(f"  ... [{len(best_draft) - 400} more chars]")


async def main() -> None:
    print("=" * 70)
    print("  Reflection Loop — Dataset: Writing Prompts (technical writing)")
    print("=" * 70)

    # Run with different thresholds to show the effect
    configs = [
        (WRITING_PROMPTS[0], 0.80),  # Moderate threshold → 1-2 iterations
        (WRITING_PROMPTS[1], 0.88),  # Strict threshold → 2-3 iterations
        (WRITING_PROMPTS[2], 0.75),  # More permissive threshold → 1 iteration
    ]

    for prompt, threshold in configs:
        await run_reflection(prompt, threshold=threshold)
        print("─" * 70)

    # Demo of the @with_reflection decorator
    print("\n[@with_reflection decorator for LangGraph nodes]")
    print("  Usage:")
    print("    from prismal.agents.patterns.reflection import with_reflection")
    print()
    print("    @with_reflection(threshold=0.85, max_iterations=2, critique_fn=my_critic)")
    print("    async def writer_node(state: AgentState) -> str:")
    print("        # Your generation logic here")
    print("        return await llm.generate(state)")
    print()
    print("  The decorator:")
    print("    1. Wraps the node function with reflection_loop")
    print("    2. Stores reflection_score in state['metadata'][node_name]")
    print("    3. Is compatible with any LangGraph subgraph node")

    print("\n[Reflection Loop pattern summary]")
    print("  Iteration 1: initial draft                    → low score")
    print("  Iteration 2: refined with specific feedback   → medium score")
    print("  Iteration 3: refined again                    → score >= threshold")
    print("  Returns: the draft with the highest score observed")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()